# Visualize Gate MLPs (R¹→R¹)

In [ ]:
import sys
from pathlib import Path
from typing import Union

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from jaxtyping import Float, Int

from spd.models.components import (
    AnyComponent,
	AnyGate,
)
from spd.models.component_model import ComponentModel, Config
from spd.models.component_utils import calc_component_acts, calc_masks
from spd.plotting import plot_mask_vals

torch.set_grad_enabled(False)
# Set your model path here
# TODO: make this more general
# MODEL_PATH = "spd/experiments/tms/out/randrecon1.00e+00_p1.00e+00_lpsp1.00e-04_m200_sd0_lr1.00e-03_bs4096_ft40_hid10hid-layers1_20250609_151342_824/model_40000.pth"
MODEL_PATH = "../spd/experiments/tms/out/randrecon1.00e+00_p2.00e+00_lpsp1.00e-04_m50_sd0_lr1.00e-03_bs4096_ft40_hid10hid-layers1_20250617_195956_677/model_10000.pth"

DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load the decomposition model
model: ComponentModel; config: Config
model, config, _ = ComponentModel.from_pretrained(MODEL_PATH)

# Extract components and gates for plotting
components: dict[str, AnyComponent] = {
	k: v
	for k, v in model.components.items() 
	if hasattr(v, "A") and hasattr(v, "B")
} # type: ignore
gates: dict[str, AnyGate] = {
	k: v 
	for k, v in model.gates.items()
}

# Get the input shape from your TMS config
n_features: int = 40  # config.n_features  # This should be in your config
batch_shape: tuple[int] = (n_features,)  # For TMS models

input_magnitude: float = 1.0
model = model.to(DEVICE)

In [ ]:
# Generate the mask plots with single feature inputs (one-hot vectors)
figures, perm_indices = plot_mask_vals(
    model=model,
    components=components,
    gates=gates,
    batch_shape=batch_shape,
    device=DEVICE,
    input_magnitude=input_magnitude,
    plot_regular_masks=False,  # Only plot sparsity masks (red plots)
)

In [ ]:
top_k_dead_alive: int = 40  # Number of top alive and dead components
alive_indicies: Int[torch.Tensor, "top_k_dead_alive"] = perm_indices["linear1"][:top_k_dead_alive]
dead_indicies: Int[torch.Tensor, "top_k_dead_alive"] = perm_indices["linear1"][top_k_dead_alive:]

In [ ]:
# Create magnitude levels
n_magnitudes: int = 100
magnitudes: Float[torch.Tensor, "n_magnitudes"] = torch.linspace(0, 1, n_magnitudes, device=DEVICE)

# Create batch: each one-hot vector at each magnitude
batch: Float[torch.Tensor, "n_features*n_magnitudes n_features"] = torch.zeros(n_features * n_magnitudes, n_features, device=DEVICE)

for i, magnitude in enumerate(magnitudes):
    start_idx = i * n_features
    end_idx = (i + 1) * n_features
    batch[start_idx:end_idx] = torch.eye(n_features, device=DEVICE) * magnitude

In [ ]:
# Calculate component activations (A matrices applied to pre-weight acts)
pre_gate_acts = model.forward_with_pre_forward_cache_hooks(
    batch, module_names=list(model.gates.keys())
)[1]


As = {module_name: v.A for module_name, v in components.items()}
target_component_acts = calc_component_acts(
    pre_weight_acts=pre_gate_acts, As=As
)  # "input into MLP"

# Calculate gate outputs (post-gate activations)
masks, sparsity_masks = calc_masks(
    gates=gates,
    target_component_acts=target_component_acts,
    detach_inputs=False,
)

In [ ]:
# Get raw MLP outputs (before final nonlinearity)
raw_mlp_outputs = {}
for gate_name, gate in gates.items():
    gate_input = target_component_acts[gate_name]
    raw_mlp_outputs[gate_name] = gate._compute_pre_activation(gate_input)

In [ ]:
submodule = "linear1"
component_indices = alive_indicies[:10]

# Create grid
n_comp = len(component_indices)
cols = 5
rows = int(np.ceil(n_comp / cols))

fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

# Plot each component
for i, comp_idx in enumerate(component_indices):
    x = target_component_acts[submodule][:, comp_idx].cpu()
    y = raw_mlp_outputs[submodule][:, comp_idx].cpu()

    axes[i].scatter(x, y, s=1, c="black", alpha=0.1)
    axes[i].set_title(f"Component {comp_idx}")
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(-2, 2)
    # axes[i].set_ylim(-1, 1.5)
    axes[i].set_xlabel("Pre-activation")
    axes[i].set_ylabel("Raw MLP Output")
# Hide unused plots
for i in range(n_comp, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f"TMS 40-10, {submodule} MLP input-outputs. Alive Components, ", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
submodule = "linear1"
component_indices = dead_indicies[:10]

# Create grid
n_comp = len(component_indices)
cols = 5
rows = int(np.ceil(n_comp / cols))

fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

# Plot each component
for i, comp_idx in enumerate(component_indices):
    x = target_component_acts[submodule][:, comp_idx].cpu()
    y = raw_mlp_outputs[submodule][:, comp_idx].cpu()

    axes[i].scatter(x, y, s=1, c="black", alpha=0.1)
    axes[i].set_title(f"Component {comp_idx}")
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(-2, 2)
    # Ensure y-axis includes zero
    y_min, y_max = axes[i].get_ylim()
    eps = 0.01
    axes[i].set_ylim(min(y_min, 0) - eps, max(y_max, 0) + eps)
    axes[i].set_xlabel("Pre-activation")
    axes[i].set_ylabel("Raw MLP Output")
# Hide unused plots
for i in range(n_comp, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f"TMS 40-10, {submodule} MLP input-outputs. Dead Components, ", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
submodule = "linear2"
component_indices = alive_indicies[:10]

# Create grid
n_comp = len(component_indices)
cols = 5
rows = int(np.ceil(n_comp / cols))

fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

# Plot each component
for i, comp_idx in enumerate(component_indices):
    x = target_component_acts[submodule][:, comp_idx].cpu()
    y = raw_mlp_outputs[submodule][:, comp_idx].cpu()

    axes[i].scatter(x, y, s=1, c="black", alpha=0.1)
    axes[i].set_title(f"Component {comp_idx}")
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(-2, 2)
    # axes[i].set_ylim(-1, 1.5)
    axes[i].set_xlabel("Pre-activation")
    axes[i].set_ylabel("Raw MLP Output")
# Hide unused plots
for i in range(n_comp, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f"TMS 40-10, {submodule} MLP input-outputs. Alive Components, ", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
submodule = "linear2"
component_indices = dead_indicies[:10]

# Create grid
n_comp = len(component_indices)
cols = 5
rows = int(np.ceil(n_comp / cols))

fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

# Plot each component
for i, comp_idx in enumerate(component_indices):
    x = target_component_acts[submodule][:, comp_idx].cpu()
    y = raw_mlp_outputs[submodule][:, comp_idx].cpu()

    axes[i].scatter(x, y, s=1, c="black", alpha=0.1)
    axes[i].set_title(f"Component {comp_idx}")
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(-2, 2)
    # Ensure y-axis includes zero
    y_min, y_max = axes[i].get_ylim()
    eps = 0.01
    axes[i].set_ylim(min(y_min, 0) - eps, max(y_max, 0) + eps)
    axes[i].set_xlabel("Pre-activation")
    axes[i].set_ylabel("Raw MLP Output")
# Hide unused plots
for i in range(n_comp, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f"TMS 40-10, {submodule} MLP input-outputs. Dead Components, ", fontsize=16)
plt.tight_layout()
plt.show()